# User Management

## Import Libraries

In [6]:
import os
import re
import sqlite3
import tkinter as tk
import uuid
from enum import Enum
from tkinter import messagebox

import pandas as pd
import ttkbootstrap as ttk

from ipynb.fs.full.base_frame import BaseFrame

## User Type Enum

In [7]:
class UserType(Enum):
    """Enumerates supported user profiles with numeric index and label."""

    PROFESSOR = (1, "Professor")
    STUDENT = (2, "Student")
    VISITOR = (3, "Visitor")

    @property
    def index(self) -> int:
        """Return numeric index for stable ordering and persistence."""

        return self.value[0]

    @property
    def label(self) -> str:
        """Return display label used in UI and filtering."""

        return self.value[1]

    @classmethod
    def labels(cls) -> list:
        """Return all labels preserving enum declaration order."""

        return [member.label for member in cls]

    @classmethod
    def from_label(cls, label: str):
        """Return enum member from a display label."""

        normalized = (label or "").strip().lower()
        for member in cls:
            if member.label.lower() == normalized:
                return member
        raise ValueError("Invalid user type label.")

## User Entity Class

In [8]:
class User:
    """Represents a system user record with identity and contact fields."""

    def __init__(self, name, email, address, user_id, user_type):
        """Initialize a user entity.

        Args:
            name: User full name.
            email: Unique contact email.
            address: User address information.
            user_id: Unique user identifier (UUID string).
            user_type: UserType enum member.
        """
        self.name = name
        self.email = email
        self.address = address
        self.user_id = user_id
        self.user_type = user_type

    def __str__(self):
        """Return a human-readable summary for quick inspection."""

        return f"User: {self.name}, Email: {self.email}, Type: {self.user_type.label}, ID: {self.user_id}"

    def __repr__(self):
        """Return a precise representation for debugging and tests."""

        return (
            f"User(name={self.name!r}, email={self.email!r}, address={self.address!r}, "
            f"user_id={self.user_id!r}, user_type={self.user_type.label!r})"
        )

## User Management Class

In [ ]:
# noinspection PyTypeChecker
class UserManagement(BaseFrame):
    """Manage user data, filtering, UI interactions, and persistence."""

    email_pattern = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

    def __init__(self, master=None, no_ui=False, shared_users=None):
        """Initialize in-memory state and optional Tkinter/UI bindings.

        Args:
            master: Parent Tk widget used when rendering the feature.
            no_ui: If True, skip UI objects to support unit tests.
        """
        self.users = shared_users if shared_users is not None else self._default_user_groups()
        self.emails = set()
        self.refresh_email_index()

        if not no_ui:
            self.editing_original_id = None
            self.editing_row_index = None
            self.edit_name_var = tk.StringVar()
            self.edit_email_var = tk.StringVar()
            self.edit_address_var = tk.StringVar()
            self.edit_type_var = tk.StringVar(value=UserType.STUDENT.label)
            self.filter_name_var = tk.StringVar()
            self.filter_email_var = tk.StringVar()
            self.filter_type_var = tk.StringVar(value="All")
            super().__init__(master)
            self.refresh_table_from_data()

    @staticmethod
    def _default_user_groups() -> dict:
        """Return the default mapping from user type to user list."""

        return {user_type.label: [] for user_type in UserType}

    def refresh_email_index(self) -> None:
        """Rebuild the email set from current user collections."""

        self.emails = {
            str(user.email).strip().lower()
            for users in self.users.values()
            for user in users
            if isinstance(user, User) and str(user.email).strip()
        }

    def has_duplicate_email(self, email: str, exclude_user: User = None) -> bool:
        """Check if an email already exists, optionally ignoring one user object.

        Args:
            email: Candidate email.
            exclude_user: Optional current user ignored during comparisons.

        Returns:
            bool: True when another user with the same email exists.
        """
        normalized_email = (email or "").strip().lower()
        if not normalized_email:
            return False

        if normalized_email not in self.emails:
            return False

        if exclude_user is None:
            return True

        if normalized_email != str(exclude_user.email).strip().lower():
            return True

        for users in self.users.values():
            for current_user in users:
                if not isinstance(current_user, User):
                    continue
                if current_user is exclude_user:
                    continue
                if str(current_user.email).strip().lower() == normalized_email:
                    return True

        return False

    def add_user(self, user_type: str, user: User):
        """Add a user to a type group after validating email uniqueness."""

        if user_type not in self.users:
            self.users[user_type] = []

        if self.has_duplicate_email(user.email):
            raise ValueError("A user with this email already exists.")

        self.users[user_type].append(user)
        self.emails.add(str(user.email).strip().lower())

    def filter_users(self, user_type: str = "All", name: str = "", email: str = "") -> dict:
        """Filter user groups by type and optional name/email fragments."""

        selected_type = (user_type or "All").strip()
        name_filter = (name or "").strip().lower()
        email_filter = (email or "").strip().lower()

        types_list = list(self.users.keys())
        if selected_type.lower() == "all":
            target_types = types_list
        elif selected_type in self.users:
            target_types = [selected_type]
        else:
            raise ValueError("Type must be 'All' or an existing user type.")

        filtered = {type_name: [] for type_name in target_types}
        for type_name in target_types:
            for user in self.users.get(type_name, []):
                if not isinstance(user, User):
                    continue

                if name_filter and name_filter not in str(user.name).lower():
                    continue
                if email_filter and email_filter not in str(user.email).lower():
                    continue

                filtered[type_name].append(user)

        return filtered

    # --- Base Frame UI Functions Overridden ---
    def get_title(self) -> str:
        """Return the feature title shown in the main panel."""
        return "👥 User Management"

    def build_right_panel(self) -> None:
        """Build right-side action controls for save/load operations."""

        super().build_right_panel()

        actions_frame = ttk.Frame(master=self.right_panel, padding=(10, 8))
        actions_frame.pack(fill=tk.X)

        ttk.Button(master=actions_frame, text="Save Data", bootstyle="success", command=self.save_to_sqlite).pack(fill=tk.X, pady=(0, 8))
        ttk.Button(master=actions_frame, text="Load Data", command=self.load_data).pack(fill=tk.X)

    def build_left_panel(self) -> None:
        """Build form, filter, and table sections and bind edit events."""

        super().build_left_panel()
        self.build_form()
        self.build_filter()
        self.refresh_table_from_data()
        self.create_data_table(self.left_panel_container, row=0, column=1, rowspan=3, colspan=2,
                               sticky=tk.NSEW, padx=10, pady=(10, 0), title="Users", height=528)
        self.table.bind("<Double-Button-1>", lambda _event: self.edit_selected_row())

    def clear_edit_form(self) -> None:
        """Clear all edit fields and reset row-edit tracking attributes."""

        self.edit_name_var.set("")
        self.edit_email_var.set("")
        self.edit_address_var.set("")
        self.edit_type_var.set(UserType.STUDENT.label)
        self.editing_original_id = None
        self.editing_row_index = None

    @staticmethod
    def is_valid_email(value: str) -> bool:
        """Validate basic email format."""

        if value is None:
            return False
        return re.search(UserManagement.email_pattern, value.strip()) is not None

    @staticmethod
    def validate_email_field(value: str) -> bool:
        """Validate email input during key-by-key UI typing."""

        if value == "":
            return True
        return " " not in value

    def save_changes(self) -> None:
        """Create a new user or update an existing one from form values."""

        name = self.edit_name_var.get().strip()
        email = self.edit_email_var.get().strip().lower()
        address = self.edit_address_var.get().strip()
        selected_type = self.edit_type_var.get().strip()

        if not name:
            messagebox.showerror("Validation Error", "Name is required.")
            return
        if not email:
            messagebox.showerror("Validation Error", "Email is required.")
            return
        if not UserManagement.is_valid_email(email):
            messagebox.showerror("Validation Error", "Invalid email format.")
            return
        if not address:
            messagebox.showerror("Validation Error", "Address is required.")
            return
        if selected_type not in self.users:
            messagebox.showerror("Validation Error", "Invalid user type.")
            return

        is_editing = self.editing_original_id is not None and self.editing_row_index is not None

        if is_editing:
            target_user = None
            source_type = None
            source_index = None

            for type_name, users in self.users.items():
                for idx, current_user in enumerate(users):
                    if isinstance(current_user, User) and current_user.user_id == self.editing_original_id:
                        target_user = current_user
                        source_type = type_name
                        source_index = idx
                        break
                if target_user is not None:
                    break

            if target_user is None:
                messagebox.showerror("Update Error", "The selected row could not be found.")
                self.refresh_table_from_data()
                self.clear_edit_form()
                return

            if self.has_duplicate_email(email, exclude_user=target_user):
                messagebox.showerror("Validation Error", "A user with this email already exists.")
                return

            old_email = str(target_user.email).strip().lower()
            target_user.name = name
            target_user.email = email
            target_user.address = address
            target_user.user_type = UserType.from_label(selected_type)

            if selected_type != source_type:
                self.users[source_type].pop(source_index)
                self.users[selected_type].append(target_user)

            self.emails.discard(old_email)
            self.emails.add(email)
            success_message = "User updated successfully."
        else:
            user = User(name, email, address, str(uuid.uuid4()), UserType.from_label(selected_type))
            try:
                self.add_user(selected_type, user)
            except ValueError as exc:
                messagebox.showerror("Validation Error", str(exc))
                return
            success_message = "User saved successfully."

        self.refresh_table_from_data()

        if is_editing and self.editing_row_index is not None and self.table is not None:
            max_index = len(self.data) - 1
            selected_index = min(self.editing_row_index, max_index)
            if selected_index >= 0 and hasattr(self.table, "setSelectedRow"):
                self.table.setSelectedRow(selected_index)
                self.table.redraw()

        messagebox.showinfo("Success", success_message)
        self.clear_edit_form()

    def edit_selected_row(self) -> None:
        """Load selected table row values into form fields for editing."""

        if self.table is None or self.data is None or len(self.data) == 0:
            messagebox.showwarning("Selection Required", "No rows available to edit.")
            return

        row_index = self.table.getSelectedRow()
        if row_index is None or row_index < 0 or row_index >= len(self.data):
            messagebox.showwarning("Selection Required", "Select one row in the table to edit.")
            return

        row = self.data.iloc[row_index]
        name = "" if pd.isna(row["name"]) else str(row["name"]).strip()
        email = "" if pd.isna(row["email"]) else str(row["email"]).strip().lower()
        address = "" if pd.isna(row["address"]) else str(row["address"]).strip()
        user_id = "" if pd.isna(row["user_id"]) else str(row["user_id"]).strip()
        user_type = "" if pd.isna(row["type"]) else str(row["type"]).strip()

        if not user_id or user_type not in self.users:
            messagebox.showwarning("Invalid Row", "Selected row has invalid user data.")
            return

        self.edit_name_var.set(name)
        self.edit_email_var.set(email)
        self.edit_address_var.set(address)
        self.edit_type_var.set(user_type)
        self.editing_original_id = user_id
        self.editing_row_index = row_index

    def refresh_table_from_data(self) -> None:
        """Refresh the table model from the current full users dataset."""
        self.set_data(self.users_to_dataframe())

    def apply_filters(self) -> None:
        """Apply UI filters and update the table with filtered results."""

        try:
            filtered = self.filter_users(
                user_type=self.filter_type_var.get(),
                name=self.filter_name_var.get(),
                email=self.filter_email_var.get()
            )
        except ValueError as exc:
            messagebox.showerror("Filter Error", str(exc))
            return

        self.set_data(self.users_to_dataframe(filtered))

    def clear_filters(self) -> None:
        """Clear filter fields and restore full data in the table."""

        self.filter_name_var.set("")
        self.filter_email_var.set("")
        self.filter_type_var.set("All")
        self.refresh_table_from_data()

    def build_filter(self) -> None:
        """Build filter controls and filter action buttons."""

        filter_frame = tk.LabelFrame(master=self.left_panel_container, text="Filters")
        filter_frame.grid(row=2, column=0, sticky="nwe", padx=10, pady=(0, 0))

        ttk.Label(master=filter_frame, text="Name").grid(row=0, column=0, sticky=tk.W, padx=(10, 10), pady=(0, 10))
        ttk.Entry(master=filter_frame, textvariable=self.filter_name_var).grid(row=0, column=1, sticky="ew", pady=(0, 10), padx=(0, 10))

        ttk.Label(master=filter_frame, text="Email").grid(row=1, column=0, sticky=tk.W, padx=(10, 10), pady=(0, 10))
        ttk.Entry(master=filter_frame, textvariable=self.filter_email_var).grid(row=1, column=1, sticky="ew", pady=(0, 10), padx=(0, 10))

        ttk.Label(master=filter_frame, text="Type").grid(row=2, column=0, sticky=tk.W, padx=(10, 10))
        ttk.Combobox(
            master=filter_frame,
            textvariable=self.filter_type_var,
            values=["All"] + UserType.labels(),
            state="readonly"
        ).grid(row=2, column=1, sticky="ew", pady=(0, 10), padx=(0, 10))

        actions = ttk.Frame(master=filter_frame)
        actions.grid(row=3, column=0, columnspan=2, sticky="ew", padx=10, pady=(0, 10))
        ttk.Button(master=actions, text="Apply Filter", command=self.apply_filters).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 4))
        ttk.Button(master=actions, text="Clear Filter", command=self.clear_filters).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(4, 0))

        filter_frame.grid_columnconfigure(1, weight=1)

    def build_form(self) -> None:
        """Build the create/update form and associated action buttons."""

        form_frame = tk.Frame(master=self.left_panel_container)
        form_frame.grid(row=0, column=0, rowspan=2, sticky="nwe", padx=10, pady=6)

        email_validator = (self.register(UserManagement.validate_email_field), "%P")

        ttk.Label(master=form_frame, text="Edit User", style="h9.TLabel").pack(anchor=tk.W)
        sep = tk.Frame(master=form_frame, height=1)
        sep.pack(fill=tk.X, pady=(4, 12))
        sep.configure(background="#444444")

        ttk.Label(master=form_frame, text="Name").pack(anchor=tk.W)
        ttk.Entry(master=form_frame, textvariable=self.edit_name_var).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Email").pack(anchor=tk.W)
        ttk.Entry(master=form_frame, textvariable=self.edit_email_var, validate="key", validatecommand=email_validator).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Address").pack(anchor=tk.W)
        ttk.Entry(master=form_frame, textvariable=self.edit_address_var).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Type").pack(anchor=tk.W)
        ttk.Combobox(master=form_frame, textvariable=self.edit_type_var, values=UserType.labels(), state="readonly").pack(fill=tk.X, pady=(2, 12))

        actions = ttk.Frame(master=form_frame)
        actions.pack(fill=tk.X)
        ttk.Button(master=actions, text="Save Changes", bootstyle="success", command=self.save_changes).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 4))
        ttk.Button(master=actions, text="Clear", bootstyle="secondary", command=self.clear_edit_form).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(4, 0))

    def save_to_sqlite(self, db_path: str = "library.db") -> None:
        """Persist in-memory users to a SQLite database."""

        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS users (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_id TEXT NOT NULL UNIQUE,
                    name TEXT NOT NULL,
                    email TEXT NOT NULL UNIQUE,
                    address TEXT NOT NULL,
                    user_type TEXT NOT NULL
                )
            """)

            cursor.execute("DELETE FROM users")
            for type_name, users in self.users.items():
                for user in users:
                    if not isinstance(user, User):
                        continue
                    if not user.name or not user.email or not user.address:
                        continue
                    cursor.execute(
                        """
                        INSERT INTO users (user_id, name, email, address, user_type)
                        VALUES (?, ?, ?, ?, ?)
                        """,
                        (user.user_id, user.name, user.email, user.address, type_name)
                    )
            conn.commit()

    def load_data(self, db_path: str = "library.db") -> None:
        """Load users from SQLite and refresh in-memory/UI datasets."""

        proceed = messagebox.askokcancel(
            "Load Data",
            "Current data will be lost during the data load process. Do you want to continue?"
        )
        if not proceed:
            return

        default_groups = self._default_user_groups()
        if not os.path.exists(db_path):
            self.users.clear()
            self.users.update(default_groups)
            self.refresh_email_index()
            self.refresh_table_from_data()
            return

        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='users'")
            users_exists = cursor.fetchone() is not None
            if not users_exists:
                self.users.clear()
                self.users.update(default_groups)
                self.refresh_email_index()
                self.refresh_table_from_data()
                return

            cursor.execute("SELECT user_id, name, email, address, user_type FROM users ORDER BY name")
            rows = cursor.fetchall()

        loaded = self._default_user_groups()
        for user_id, name, email, address, user_type in rows:
            if user_type not in loaded:
                loaded[user_type] = []
            if not name or not email or not address or not user_id:
                continue
            try:
                enum_type = UserType.from_label(user_type)
            except ValueError:
                continue
            loaded[user_type].append(User(name, email, address, user_id, enum_type))

        self.users.clear()
        self.users.update(loaded)
        self.refresh_email_index()
        self.refresh_table_from_data()

    def users_to_dataframe(self, users: dict = None) -> pd.DataFrame:
        """Convert user mappings into a DataFrame for table rendering."""

        source = self.users if users is None else users
        rows = []
        for type_name, users in source.items():
            for user in users:
                if not isinstance(user, User):
                    continue
                rows.append({
                    "name": user.name,
                    "email": user.email,
                    "address": user.address,
                    "user_id": user.user_id,
                    "type": type_name
                })

        return pd.DataFrame(rows, columns=["name", "email", "address", "user_id", "type"])

## Unit Test

In [ ]:
import unittest

class UserManagementTest(unittest.TestCase):
    """Test user management behavior, filtering, and validation rules."""

    def test_add_user(self):
        management = UserManagement(master=None, no_ui=True)
        user = User("Joao", "joao@email.com", "Rua X", str(uuid.uuid4()), UserType.STUDENT)
        management.add_user(UserType.STUDENT.label, user)
        self.assertEqual(len(management.users[UserType.STUDENT.label]), 1)

    def test_has_duplicate_email(self):
        management = UserManagement(master=None, no_ui=True)
        user = User("Maria", "maria@email.com", "Rua Y", str(uuid.uuid4()), UserType.PROFESSOR)
        management.add_user(UserType.PROFESSOR.label, user)

        self.assertTrue(management.has_duplicate_email("maria@email.com"))
        self.assertFalse(management.has_duplicate_email("outro@email.com"))
        self.assertFalse(management.has_duplicate_email("maria@email.com", exclude_user=user))

    def test_filter_users(self):
        management = UserManagement(master=None, no_ui=True)
        management.add_user(UserType.STUDENT.label, User("Ana", "ana@email.com", "Rua 1", str(uuid.uuid4()), UserType.STUDENT))
        management.add_user(UserType.VISITOR.label, User("Bruno", "bruno@email.com", "Rua 2", str(uuid.uuid4()), UserType.VISITOR))

        filtered_by_name = management.filter_users(name="ana")
        self.assertEqual(sum(len(v) for v in filtered_by_name.values()), 1)

        filtered_by_email = management.filter_users(email="bruno@email.com")
        self.assertEqual(sum(len(v) for v in filtered_by_email.values()), 1)

        filtered_by_type = management.filter_users(user_type=UserType.STUDENT.label)
        self.assertEqual(list(filtered_by_type.keys()), [UserType.STUDENT.label])
        self.assertEqual(len(filtered_by_type[UserType.STUDENT.label]), 1)

    def test_is_valid_email(self):
        self.assertTrue(UserManagement.is_valid_email("abc@x.com"))
        self.assertFalse(UserManagement.is_valid_email("abc"))

unittest.main(argv=[''], exit=False)